# RAG System for Lord of the Rings Q&A

**Retrieval-Augmented Generation (RAG)** system for answering questions about Lord of the Rings text using Google Gemini models.

## Features:
- Document chunking and vector embeddings
- Persistent Chroma vector database
- Context-based answers with source references
- Batch and interactive question modes

## Flow:
1. Load LOTR text → chunk into 1000-char pieces
2. Create Gemini embeddings → store in Chroma DB
3. Retrieve relevant chunks → generate answers with Gemini-2.5-flash
4. Display answers with source previews

## import dependencies

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

## config

In [2]:
EMBEDDING_MODEL = "models/gemini-embedding-001"
LLM_MODEL = "gemini-2.5-flash"
PERSISTENT_DIR = "db/chroma_db"
DOCUMENT_FILE = "documents/lord_of_the_rings.txt"

## define persistent and document directories

In [ ]:
current_dir = os.getcwd()
persistent_dir = os.path.join(current_dir, PERSISTENT_DIR)
document_file = os.path.join(current_dir, DOCUMENT_FILE)

## init chroma vector database

In [4]:
if not os.path.exists(persistent_dir):
    print("Persistent directory does not exist. Initializing vector store...")

    # Ensure the text file exists
    if not os.path.exists(document_file):
        raise FileNotFoundError(
            f"The file {document_file} does not exist. Please check the path."
        )
    
    # read the content of the text file
    loader = TextLoader(document_file)
    documents = loader.load()

    # split doc into chunks
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    docs = text_splitter.split_documents(documents)

    # Display information about the split documents
    print("\n--- Document Chunks Information ---")
    print(f"Number of document chunks: {len(docs)}")
    print(f"Sample chunk:\n{docs[0].page_content}\n")

    # Create embeddings
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

    # Create the vector store and persist it 
    print("\n--- Creating vector store ---")
    db = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=persistent_dir)
    print("\n--- Finished creating vector store ---")
else:
    print("Vector store already exists. Loading existing vector store...")
    
    # Load the existing vector store
    embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
    db = Chroma(persist_directory=persistent_dir, embedding_function=embeddings)

Vector store already exists. Loading existing vector store...


## init llm

In [5]:
llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.1)

## Create a custom prompt template

In [6]:
prompt_template = """
Use the following pieces of context to answer the question at the end. 
If you don't know the answer based on the context provided, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template, 
    input_variables=["context", "question"]
)

## create a retrieval chain

In [7]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT},
    retriever=db.as_retriever(),
    return_source_documents=True
)

In [8]:
def ask_question(question):
    print(f"\n--- Asking question: {question} ---")
    result = qa_chain.invoke({"query": question})
    answer = result['result']
    source_documents = result['source_documents']
    
    print(f"\nAnswer: {answer}")
    
    if source_documents:
        print("\nSource Documents:")
        for doc in source_documents:
            print(f"- {doc.metadata.get('source', 'Unknown Source')}: {doc.page_content[:200]}...")
    else:
        print("No source documents found.")

## ask questions

In [9]:
questions = [
    "Who is the Ring-bearer?",
    "Where does Gandalf meet Frodo?",
    "What is the Fellowship of the Ring?",
    "Who is Aragorn?",
    "What happens at Mount Doom?"
]

print("\n=== Starting Q&A Session ===")
for question in questions:
    ask_question(question)


=== Starting Q&A Session ===

--- Asking question: Who is the Ring-bearer? ---

Answer: Frodo Baggins is the Ring-bearer.

Source Documents:
- /Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents/lord_of_the_rings.txt: Gandalf spoke of the history of the One Ring and the danger that it posed, but the decision was made. Frodo Baggins, the most unlikely of heroes, was chosen to be the bearer of the One Ring, a respons...
- /Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents/lord_of_the_rings.txt: It was in the heart of Hobbiton, at the home of Frodo Baggins, that Gandalf came to speak of matters far greater than Frodo could have imagined. This was the moment that Frodo learned of the perilous ...
- /Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/Langchain/RAG/documents/lord_of_the_rings.txt: Frodo stood at the edge of the Shire, holding the Ring in his hand. The weight of

In [ ]:
print("\n=== Interactive Mode ===")
print("Enter your questions (type 'q' to exit):")
while True:
    user_question = input("\nYour question: ")
    if user_question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    ask_question(user_question)